# HAM10000 Bounding-Box + Label Prep
Set the paths below, run the cells, and you'll get a new CSV that
contains every bounding box from `ham10000_bboxes.csv` plus the
diagnosis label (`dx`) pulled from `HAM10000_metadata.csv`.

In [ ]:
from pathlib import Path
import pandas as pd

IMAGES_DIR = Path("HAM10000_images")  # optional, for reference
BBOX_CSV = Path("ham10000_bboxes.csv")
METADATA_CSV = Path("HAM10000_metadata.csv")
OUTPUT_CSV = Path("ham10000_bboxes_with_labels.csv")

for p in (BBOX_CSV, METADATA_CSV):
    if not p.exists():
        raise FileNotFoundError(f'{p} not found. Update the path above.')

OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
# ---- Load CSVs --------------------------------------------------------
metadata = pd.read_csv(METADATA_CSV, dtype=str)
boxes = pd.read_csv(BBOX_CSV, dtype=str)

metadata['image_id'] = metadata['image_id'].str.strip()
boxes['image_id'] = boxes['image_id'].str.strip()

metadata[['image_id', 'dx']].head()

In [ ]:
# ---- Merge dx labels into bounding boxes ------------------------------
metadata_labels = metadata[['image_id', 'dx']].drop_duplicates('image_id')
merged = boxes.merge(metadata_labels, on='image_id', how='left', validate='m:1')
missing = merged['dx'].isna().sum()
if missing:
    raise ValueError(f'{missing} bounding boxes lack metadata labels. Check the CSVs.')

merged.head()

In [ ]:
# ---- Optional: sanity check class distribution -----------------------
class_counts = merged['dx'].value_counts().sort_index()
class_counts

In [ ]:
# ---- Save the labeled CSV -------------------------------------------
merged.to_csv(OUTPUT_CSV, index=False)
print(f'Saved labeled CSV to {OUTPUT_CSV.resolve()} (rows={len(merged)})')